## ĐẠI SỐ TUYẾN TÍNH CHO AI
### Bài 1 - Biểu diễn dữ liệu thành ma trận & độ tương đồng

> Chuẩn bị dữ liệu:

In [ ]:
import numpy as np                  # Cài đặt thư viện numpy
import matplotlib.pyplot as plt     # Cài đặt thư viên matplotlib
# 8 câu tiếng anh
sentences = [
    "I really like this subject",
    "I do no like this subject",
    "The cat chased the mouse",
    "The mouse chased the cat",
    "You can deposite your money at the bank",
    "You often sit down on the river bank to relax",
    "I really like playing football in this park",
    "I like go shopping this Sunday",
]

> Biến dữ liệu thành ma trận:

In [ ]:
# Tạo tập hợp gồm các từ xuất hiện trong các câu (Không lặp và được sắp xết theo alphabet)
set_vocab = sorted({w for s in sentences for w in s.lower().split()})
# Hàm chuyển đổi một câu thành vector
def sentences_to_vector(s):
    v = np.zeros(len(set_vocab)) # Vector toàn giá trị 0, có độ dài bằng set_vocab
    for i in s.lower().split():
        v[set_vocab.index(i)] += 1
    return v
# Tạo ma trận gồm các vector ứng với mỗi câu
X = np.array([sentences_to_vector(s) for s in sentences])   # X.shape (8 , 32)
print("X.Shape =" , X.shape , ", X is Bag-of-Words Matrix")

#### PHÂN TÍCH:
* Tổng số hàng bằng với tổng số câu cho trước. Mỗi hàng đại diện cho một vector ứng với mỗi câu.
* Tổng số cột bằng với tổng số phần tử thuộc set_vocab:
    + Mỗi cột đại diện cho số lần xuất hiện của một từ trong set_vocab.
    + Ô [i , j] có giá trị là k có nghĩa: Từ thứ j trong set_vocab xuất hiện k lần trong câu thứ i.

> Phép toán cơ bản:

In [ ]:
# Tính vector trung bình theo cột (Trung bình tần số xuất hiện của một từ trong tất cả các câu)
X_mean = X.mean(axis = 0)
print("X_mean.Shape =" , X_mean.shape)  # X_mean.Shape = (32 , )
print("X.Shape =" , X.shape)    # X.Shape = (8 , 32)
# Trừ trung bình (broadcasting) - Căn giữa dữ liệu
# Quy tắc broadcasting: Numpy tự động nhân bản X_mean để tạo ma trận (8 , 32) 
X_center = X - X_mean 
print("X_center.Shape =" , X_center.shape)     # X_center.shape = (8 , 32)

> COSINE SIMILARITY:
#### Cosine similarity đo độ tương đồng về hướng giữa hai vector, mà không phụ thuộc độ dài câu.

In [ ]:
def cosine_similarity(X , Y = None):
    if Y is None:
        Y = X       # Tính độ tương đồng về hướng của các vector với chính nó và các vector khác cùng một ma trận
    
    # Chuẩn hoá các vector ma trận theo hàng
    Xn = X / np.linalg.norm(X , axis = 1 , keepdims = True)     # Xn.Shape = (Số hàng của X, Số cột của X)
    Yn = Y / np.linalg.norm(Y , axis = 1 , keepdims = True)     # Xn.Shape = (Số hàng của Y, Số cột của Y)
    return Xn @ Yn.T    # Phép nhân ma trận Xn cho ma trận Yn chuyển vị
S = cosine_similarity(X)        # X.Shape = (8 , 32)
print("S.Shape =" , S.shape)    # S.Shape = (8 , 8)
for row in S:
    for col in row:
        print(f"{col:.3f}" , end = " ")
    print()

> Truy vấn:

In [ ]:
def Search(query , top_k = 3):
    v_query = sentences_to_vector(query).reshape(1 , -1)    # v_query.Shape = (1 , 32) , X.Shape = (8 , 32)
    sims = cosine_similarity(v_query , X).flatten()         # sims.Shape = (1 , 8)

    # np.argsort trả về mảng chứa vị trí ứng với phần tử của mảng gốc có giá trị từ nhỏ đến lớn
    idx = np.argsort(sims)[-top_k:][::-1]       # idx.Shape = (1 , top_k) (idx = np.argsort(-sims)[:top_k])
    return [(sentences[i] , sims[i]) for i in idx]
query_test = "like"
result_search = Search(query_test , 3)
print("Truy vấn từ khoá:" , query_test)
for most_similar_sentence , cosine_similarity_value in result_search:
    print(f"\t+ {most_similar_sentence} - {cosine_similarity_value:.3}")

#### Nhận xét chung:
* Ta nhận thấy rằng kết quả thu đầu rất trực quan, và đây cũng là hạn chế của kỹ thuật:
    + Hai câu:  "I really like this subject",
                "I do no like this subject"
    có nhiều từ chung: "I", "like", "this", "subject" nên có giá trị cosine_similarity cao (0.730)
    nhưng thực tế, cả hai câu lại trái nghĩa nhau.
    + Hai câu:  "The cat chased the mouse",
                "The mouse chased the cat"
    được cấu thành từ một tập từ giống hệt nhau nên có giá trị cosine_similarity bằng 1.000
    nhưng thực tế, hai câu có ý nghĩa ngược nhau.
* Đổi lại, kỹ thuật cosine_similarity không phụ thuộc vào độ dài câu, giải quyết vấn đề vector thừa
và kết quả được chuẩn hoá nên đóng vai trò lớn trong bài toán "Truy vấn"

### Bài 2 - Biến đổi tuyến tính & SVD

> Giảm chiều SVD:

In [ ]:
U , S_svd , V_t = np.linalg.svd(X_center , full_matrices = False)
coords = U[:,:2] * S_svd[:2] 
print("U.Shape" , "S_svd.Shape" , "V_t.shape" , "coords.shape" , sep = ", " , end = " = ")
print(U.shape , S_svd.shape , V_t.shape , coords.shape , sep = " , ")     #(8,8) (8,) (8,32) (8,2)
print("Toạ độ của các câu:")
index_coords = 1
for row in coords:
    print(f"Câu {index_coords}:" , end = " ")
    index_coords += 1
    print(f"({row[0]:.3f} , {row[1]:.3f})")

> Vẽ mô phỏng

In [ ]:
# Vẽ mô phỏng:
plt.figure(figsize = (8 , 6))  # Tạo khung 8x6 inch
colors = ["tab:blue"] * 2 + ["tab:pink"] * 2 + ["tab:red"] * 1 + ["tab:green"] * 1+ ["tab:blue"] * 2
plt.scatter(coords[: , 0] , coords[: , 1] , c = colors , s = 120 , edgecolors = "black")
for i , (x , y) in enumerate(coords):
    plt.annotate(f"" , (x , y) , xytext = (10 , 10) , textcoords="offset points" , fontsize = 10)

plt.axhline(0, color="red", linewidth=0.5)
plt.axvline(0, color="red", linewidth=0.5)
plt.xlabel("Trục Ox")
plt.ylabel("Trục Oy")
plt.title("Biểu diễn 8 câu trên không gian 2D sau khi giảm chiều bằng SVD")
plt.legend(handles=[
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue", markersize=10, label="Chủ đề Sở thích"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:pink", markersize=10, label="Chủ đề Động vật"),
])
plt.grid(alpha = 0.3)
plt.tight_layout()
plt.show()

#### Nhận xét chung:
* Bốn nút xanh đại diện cho câu 1, 2, 7, 8 (chung chủ đề Sở thích): đứng gần nhau
* Hai nút hồng đại diện cho câu 3, 4 (chung chủ đề Động vật): trùng nhau hoàn toàn (hai câu được cấu thành từ các từ giống nhau và trùng cả về số lượng tương ứng)
* Các nút còn lại không liên quan với nhau và không liên quan với hai chủ đề trên nên nằm rải rác

### BỘ PHÂN LOẠI 1-NN DỰA TRÊN COSINE

>Bộ phân loại 1-NN dựa trên cosine sẽ nhận một câu truy vấn mới, tính độ tương đồng cosine giữa câu này với tất cả các câu đã có trong cơ sở dữ liệu, tìm ra câu có độ tương đồng lớn nhất, và lấy nhãn của câu đó gán cho câu truy vấn.

In [ ]:
# nhãn thật của 8 câu
sentences_label = [
    "Sở thích",
    "Sở thích",
    "Động vật",
    "Động vật",
    "Hướng dẫn 1",
    "Hướng dẫn 2",
    "Sở thích",
    "Sở thích"
]
def sentences_to_vector_safe(s):
    v = np.zeros(len(set_vocab))
    for i in s.lower().split():
        if i in set_vocab:          # chỉ cộng khi từ tồn tại trong set_vocab
            v[set_vocab.index(i)] += 1
    return v

def classify_1nn(query):
    query_vector = sentences_to_vector_safe(query).reshape(1, -1)
    sims = cosine_similarity(query_vector, X).flatten()
    # np.argmax(): trả về index của phần tử có giá tị lớn nhất
    idx_nearest_meaning = np.argmax(sims)   
    return sentences_label[idx_nearest_meaning], sims[idx_nearest_meaning]

test_sentence = "I like watching Ronaldo"
topic , nearly_score = classify_1nn(test_sentence)
print(f"Câu: \"{test_sentence}\"")
print(f"Dự đoán nhãn: {topic}  (Ứng với giá trị cosine_similarity: {nearly_score:.3f})")